# 🎓 Course Analytics: The Live Analysis

Welcome to Part 2 of the data story! In this notebook, we're going to compare two completely different courses by looking at survey data side-by-side.

We have data from:
- **Course A**: Econometrics (known for being strict and math-heavy)
- **Course B**: History of Economics (known for being chill and reading-heavy)

### What we're trying to figure out:
1. **Return on Investment (ROI):** How many hours of studying does it take to get a good grade in each class? Is the effort "worth it"?
2. **The Teacher Matrix:** We'll plot "Strictness" vs. "Likability" to categorize professors into four quadrants (like *The Snoopzefest* or *The Masterclass*).
3. **Incentive Structures:** Does a strict teacher *actually* force students to study more? We'll run a statistical test to find out for sure.

---

## 🔧 Setup & Data Cleaning

First, we load the data from two separate CSV files (one for each course). But raw survey data is usually messy! 

The column names from Google Forms are often full sentences like _"How many hours per week do you study for this course outside of class?"_. That's too long to type out every time we want to plot a graph. 

So, as a classic data analyst move, **we rename the columns on the fly** into neat, one-word variables right after combining the datasets.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120

# 1. Load Data
df_a = pd.read_csv('Course_A_simulated.csv')
df_b = pd.read_csv('Course_B_simulated.csv')

# 2. Add Course Labels before combining
df_a['Course Name'] = 'Course A (Econometrics)'
df_b['Course Name'] = 'Course B (History of Econ)'

# 3. Combine them into one Dataframe
df = pd.concat([df_a, df_b])

# 4. Clean up the verbose survey column names for easier coding and plotting
rename_map = {
    "How many hours per week do you study for this course outside of class?": "Study_Hours",
    "What percentage grade (0-100) do you realistically expect to get?": "Expected_Grade",
    "On a scale of 1 to 10, how strict/demanding is the professor?": "Strictness",
    "On a scale of 1 to 10, how much do you actually enjoy their lectures?": "Likability"
}
df.rename(columns=rename_map, inplace=True)

# Let's take a peak at our newly cleaned combined dataset!
df.head()

---

## 📈 View 1: The "ROI" Scatter Plot (Effort vs. Grade)

### What this does
We plot **Study Hours** on the bottom (X-axis) against the **Expected Grade** on the side (Y-axis).

### In plain English
This shows you the "Return on Investment" for your studying.
- In **Course A (Blue)**, notice how the line slopes gently. To get a high grade, you have to put in *a lot* of hours. The curve might even flatten out — meaning studying for 20 hours instead of 15 barely bumps up your grade.
- In **Course B (Orange)**, the line likely shoots up quickly. A few hours of studying translates directly to a high expected grade.

The trend lines (called LOWESS curves) show the *average path* through the chaotic scatter of dots, making it easier to see the overall trend.

In [ ]:
plt.figure(figsize=(10, 6))

# Plot scatter points
sns.scatterplot(x='Study_Hours', y='Expected_Grade', hue='Course Name', data=df, alpha=0.7, s=60, edgecolor='white')

# Overlay lowess trend lines to show the curve of returns
sns.regplot(x='Study_Hours', y='Expected_Grade', data=df[df['Course Name']=='Course A (Econometrics)'], 
            scatter=False, lowess=True, color='#2563eb', line_kws={'linewidth': 2})
sns.regplot(x='Study_Hours', y='Expected_Grade', data=df[df['Course Name']=='Course B (History of Econ)'], 
            scatter=False, lowess=True, color='#f59e0b', line_kws={'linewidth': 2})

plt.title("Return on Investment (ROI): Effort vs. Expected Grade", fontsize=14, fontweight='bold')
plt.xlabel("Study Hours per Week", fontsize=12)
plt.ylabel("Expected Grade (%)", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

---

## 🎯 View 2: The Teacher Matrix (Strictness vs. Likability)

### What this does
We map each student's rating of their professor based on two axes: **Strictness/Demanding** (X-axis) and **Likability/Enjoyment** (Y-axis). 

We also add a third dimension: **Bubble Size**. The larger the bubble, the more hours that student studies.

### In plain English
This divides the classroom experience into four distinct quadrants:
1. **The Fun & Easy (Top Left):** Low strictness, high likability. (Usually packed with huge waitlists!)
2. **The Tough Love (Top Right):** High strictness, but you highly enjoy the lectures. (The "Masterclass" — hard work, but worth it.)
3. **The Snoozefest (Bottom Left):** Low strictness, but incredibly boring.
4. **The Tyrant (Bottom Right):** High strictness, and you hate every minute of it.

Watch where the bubbles are biggest — it shows us *where* students are putting in the most effort!

In [ ]:
plt.figure(figsize=(10, 6))
# Using bubble size to represent Study Hours
sns.scatterplot(x='Strictness', y='Likability', hue='Course Name', size='Study_Hours', 
                sizes=(20, 400), data=df, alpha=0.7, edgecolor='black', linewidth=0.5)

# Draw crosshairs (dividing at the mid-point of a 1-10 scale)
plt.axvline(x=5.5, color='gray', linestyle='--')
plt.axhline(y=5.5, color='gray', linestyle='--')

# Annotate Quadrants
plt.text(2, 9, "The Fun & Easy", fontsize=13, color='#10b981', alpha=0.8, weight='bold')
plt.text(8, 9, "The Tough Love", fontsize=13, color='#2563eb', alpha=0.8, weight='bold')
plt.text(2, 2, "The Snoozefest", fontsize=13, color='#6b7280', alpha=0.8, weight='bold')
plt.text(8, 2, "The Tyrant", fontsize=13, color='#dc2626', alpha=0.8, weight='bold')

plt.title("The Teacher Matrix: Strictness vs. Likability\n(Bubble Size = Study Hours)", fontsize=14, fontweight='bold')
plt.xlabel("Strictness (1 = Chill, 10 = Terrifying)", fontsize=12)
plt.ylabel("Likability (1 = Snoozefest, 10 = Masterclass)", fontsize=12)
plt.xlim(0.5, 10.5)
plt.ylim(0.5, 10.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

---

## 🧪 View 3: Live Hypothesis Test

### The Question
We can *see* that students study differently in Course A vs Course B. But is that difference statistically significant? Or did we just happen to poll a few lazy students in Course B by coincidence?

### What this does
We run an **Independent T-Test**. It compares the average study hours in Course A against the average in Course B.

### In plain English
A Hypothesis Test gives us a **P-value**. Think of the P-value as a *probability of coincidence*.
- The "Null Hypothesis" is the boring assumption: "There is no real difference between these courses."
- If our P-value is **less than 0.05** (less than a 5% chance of coincidence), we strongly reject that boring assumption. We can confidently say: *"Yes, the tough nature of Course A is forcing students to study significantly more!"*
- If our P-value is **greater than 0.05**, we fail to formally prove a difference. 

In [ ]:
# Filter the study hours for both courses
course_a_hours = df[df['Course Name'] == 'Course A (Econometrics)']['Study_Hours']
course_b_hours = df[df['Course Name'] == 'Course B (History of Econ)']['Study_Hours']

# Run the T-Test
t_stat, p_val = stats.ttest_ind(course_a_hours, course_b_hours)

print("=" * 60)
print("  HYPOTHESIS TEST: Does strictness drive higher effort?")
print("=" * 60)
print(f"Course A (Strict) Avg Study Hours: {course_a_hours.mean():.1f} hours/week")
print(f"Course B (Chill)  Avg Study Hours: {course_b_hours.mean():.1f} hours/week")
print(f"\nT-Statistic: {t_stat:.3f}")
print(f"P-Value:     {p_val:.5f}")
print("=" * 60)

if p_val < 0.05:
    print("\n✅ CONCLUSION: Reject the Null Hypothesis.")
    print("The data mathematically proves that strictness drives significantly higher effort.")
else:
    print("\n❌ CONCLUSION: Fail to Reject the Null Hypothesis.")
    print("Strictness does not appear to be an effective incentive statistically based on this data.")

---

### 💡 Pro Tip for Data Analysts

In the first cell, we used `.rename(columns=...)`. When analyzing raw survey data (like Google Forms), your raw column names are almost always massive, unmanageable sentences (e.g., *"On a scale of 1 to 10..."*). 

Mapping them carefully into short, snake_case strings (`Study_Hours`, `Strictness`) the very second you load them is the best way to prevent your code from becoming an unreadable mess!

---

*Notebook prepared for the ProofGrad analytical demonstration series.*